# Fine Tuning BERT for Sentiment Analysis with PyTorch

References:
- [Fine Tuning BERT for Sentiment Analysis with PyTorch](https://wellsr.com/python/fine-tuning-bert-for-sentiment-analysis-with-pytorch/)
- [Twitter US Airline Sentiment](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment?select=Tweets.csv)

- [RuntimeError: CUDA out of memory during loss.backward()](https://discuss.pytorch.org/t/runtimeerror-cuda-out-of-memory-during-loss-backward/53450/6)
- [Forms - Colaboratory](https://colab.research.google.com/notebooks/forms.ipynb)
- [Plotting loss curve](https://discuss.pytorch.org/t/plotting-loss-curve/42632/5)
- [Estratégias eficazes para lidar com conjuntos de dados desbalanceados](https://medium.com/@daniele.santiago/estrat%C3%A9gias-eficazes-para-lidar-com-conjuntos-de-dados-desbalanceados-5b873894483b)
- [Aprenda a balancear seus dados com Undersampling e Oversampling em Python](https://medium.com/@daniele.santiago/aprenda-a-balancear-seus-dados-com-undersampling-e-oversampling-em-python-6fd87095d717#:~:text=J%C3%A1%20o%20over%2Dsampling%2C%20por,t%C3%A9cnicas%20foi%20explicitada%20neste%20artigo.)
- [RandomOverSampler](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html#imblearn.over_sampling.RandomOverSampler.fit_resample)
- [REPRODUCIBILITY](https://pytorch.org/docs/stable/notes/randomness.html)
- [Análise de sentimentos em português utilizando Pytorch e Python](https://medium.com/data-hackers/an%C3%A1lise-de-sentimentos-em-portugu%C3%AAs-utilizando-pytorch-e-python-91a232165ec0)

In [1]:
# @title Environment running
running_local = False # @param {type:"boolean"}
if running_local:
    running_colab = running_kaggle = False
else:
    running_colab = True  # @param {type:"boolean"}
    running_kaggle = not running_colab  # @param {type:"boolean"}

In [2]:
if running_colab:
    from google.colab import drive

    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Importing Required Libraries

In [3]:
%%capture
if running_colab:
    !pip install evaluate

In [50]:
import random
import time

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import wandb
from datasets import Dataset
from imblearn.over_sampling import RandomOverSampler
from huggingface_hub import notebook_login
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    pipeline,
)

## Config

In [5]:
RANDOM_SEED = 103
TEST_SIZE = 0.2
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3
MINORITY_CLASS_OVERSAMPLE_FACTOR = 3

TOKEN_MAX_LENGTH = 512
BERTIMBAU_MODEL_PATH = "neuralmind/bert-base-portuguese-cased"

SAMPLE_REVIEWS = [
    "Empresa boa para trabalhar", # positive
    "Empresa ruim para trabalhar", # negative
    "Não tenho o que declarar", # neutral
]


if running_colab:
    GLASSDOOR_MODEL_PATH = "/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/train_model/bertimbau-glassdoor-reviews-epoch_5.bin"
    GLASSDOOR_FREEZING_MODEL_PATH = "/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/train_model/bertimbau-glassdoor-reviews-freezing-epoch_5.bin"
    PATH_TO_SAVE_MODEL = "/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/train_model/"
if running_kaggle:
    GLASSDOOR_MODEL_PATH = "/kaggle/working/bertimbau-glassdoor-reviews-epoch_5.bin"
    GLASSDOOR_FREEZING_MODEL_PATH = (
        "/kaggle/working/bertimbau-glassdoor-reviews-freezing-epoch_5.bin"
    )
    PATH_TO_SAVE_MODEL = "/kaggle/working/"
if running_local:
    GLASSDOOR_MODEL_PATH = "./bertimbau-glassdoor-reviews-epoch_5.bin"
    GLASSDOOR_FREEZING_MODEL_PATH = "./bertimbau-glassdoor-reviews-freezing-epoch_5.bin"
    PATH_TO_SAVE_MODEL = "./train_model/"

In [6]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"There are {torch.cuda.device_count()} GPU(s) available.")
    print("Device name:", torch.cuda.get_device_name(0))
else:
    print("No GPU available, using the CPU instead.")
    device = torch.device("cpu")

There are 1 GPU(s) available.
Device name: Tesla T4


In [7]:
torch.manual_seed(RANDOM_SEED)

In [8]:
random.seed(RANDOM_SEED)

In [9]:
np.random.seed(RANDOM_SEED)

## Importing and Preprocessing the Dataset

In [10]:
if running_colab:
    dataset = pd.read_csv(
        "/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/data_preparation/glassdoor_reviews_annotated.csv"
    )
else:
    if running_kaggle:
        dataset = pd.read_csv(
            "/kaggle/input/glassdoor-reviews-annotated/glassdoor_reviews_annotated.csv"
        )
    else:
        dataset = pd.read_csv("../data_preparation/glassdoor_reviews_annotated.csv")

In [11]:
dataset.head(2)

,review_id,company,employee_role,employee_detail,review_text,review_date,star_rating,sentiment,annotated
0,82630669,Tecnomapas,Recepcionista,"Ex-funcionário(a), mais de um ano","Companheirismo entre os colegas, oportunidade ...",2023-12-15,5.0,1,0
1,82630669,Tecnomapas,Recepcionista,"Ex-funcionário(a), mais de um ano",Não tive nenhum ponto negativo,2023-12-15,5.0,0,1


In [12]:
filtered_dataset = dataset.filter(["review_text", "sentiment"])

In [13]:
filtered_dataset.shape

(2532, 2)

In [14]:
filtered_dataset["sentiment"].value_counts()

,count
sentiment,
1,1269
-1,1021
0,242


In [15]:
num_labels = len(filtered_dataset["sentiment"].value_counts())

In [16]:
num_labels

3

In [17]:
filtered_dataset.head()

,review_text,sentiment
0,"Companheirismo entre os colegas, oportunidade ...",1
1,Não tive nenhum ponto negativo,0
2,Equipe bem prestativa e ótima de se trabalhar.,1
3,Modo home office ainda tem que ser melhorado.,-1
4,Única vantagem era o trabalho ser home office,0


Replace negative sentiment (-1) to 2, to avoid PyTorch errors.

In [18]:
filtered_dataset["sentiment"] = filtered_dataset["sentiment"].apply(
    lambda x: 2 if x == -1 else x
)

In [19]:
filtered_dataset.head()

,review_text,sentiment
0,"Companheirismo entre os colegas, oportunidade ...",1
1,Não tive nenhum ponto negativo,0
2,Equipe bem prestativa e ótima de se trabalhar.,1
3,Modo home office ainda tem que ser melhorado.,2
4,Única vantagem era o trabalho ser home office,0


Shuffle the data before splitting it into training and testing sets to remove any inherent ordering in the data that could bias the model's learning process.

In [20]:
filtered_dataset = filtered_dataset.sample(frac=1, random_state=RANDOM_SEED)

In [21]:
train_data, test_data = train_test_split(
    filtered_dataset, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

In [22]:
train_data.shape

(2025, 2)

In [23]:
test_data.shape

(507, 2)

### Oversampling

In [24]:
def oversampling(train_df, minority_class=0, majority_classes=[1, 2]):
    # Desired number of samples for the minority class
    oversample_size = (
        train_df[train_df["sentiment"] == 0]["sentiment"].value_counts()[0]
        * MINORITY_CLASS_OVERSAMPLE_FACTOR
    )

    oversampler = RandomOverSampler(sampling_strategy={minority_class: oversample_size})
    X_res, y_res = oversampler.fit_resample(
        train_df[["review_text"]], train_df["sentiment"]
    )

    y_res_df = pd.DataFrame({"sentiment": y_res})
    resampled_df = pd.concat([X_res, y_res_df], axis=1)
    resampled_df.reset_index(drop=True, inplace=True)

    resampled_minority_df = resampled_df[resampled_df["sentiment"] == minority_class]

    # Concatenate the oversampled minority class with the majority class
    oversampled_df = pd.concat([train_df, resampled_minority_df])

    # Shuffle the DataFrame to randomize the order of samples
    oversampled_df = oversampled_df.sample(
        frac=1, random_state=RANDOM_SEED
    ).reset_index(drop=True)
    return oversampled_df

In [25]:
filtered_dataset.shape

(2532, 2)

In [26]:
filtered_dataset["sentiment"].value_counts()

,count
sentiment,
1,1269
2,1021
0,242


In [27]:
oversampled_filtered_dataset = oversampling(filtered_dataset)

In [28]:
oversampled_filtered_dataset.shape

(3258, 2)

In [29]:
oversampled_filtered_dataset["sentiment"].value_counts()

,count
sentiment,
1,1269
2,1021
0,968


In [30]:
oversampled_train_data, oversampled_test_data = train_test_split(
    oversampled_filtered_dataset, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

In [31]:
train_dataset = Dataset.from_pandas(oversampled_train_data)
val_dataset = Dataset.from_pandas(oversampled_test_data)

## Tokenize the Data

In [32]:
tokenizer = AutoTokenizer.from_pretrained(BERTIMBAU_MODEL_PATH)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [33]:
def tokenize_function(examples):
    return tokenizer(
        examples["review_text"],
        padding="max_length",
        truncation=False,
        max_length=TOKEN_MAX_LENGTH
    )

In [34]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/2606 [00:00<?, ? examples/s]

In [35]:
tokenized_val = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/652 [00:00<?, ? examples/s]

In [36]:
tokenized_train = tokenized_train.rename_column("sentiment", "labels")

In [37]:
tokenized_val = tokenized_val.rename_column("sentiment", "labels")

In [38]:
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

## Define the Model

In [39]:
model = AutoModelForSequenceClassification.from_pretrained(
    BERTIMBAU_MODEL_PATH,
    num_labels=3,
    id2label={0: "neutral", 1: "positive", 2: "negative"},
    label2id={"neutral": 0, "positive": 1, "negative": 2},
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Configure Training Arguments

In [40]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Define Evaluation Metrics

In [41]:
accuracy_metric = evaluate.load("accuracy")

In [42]:
f1_metric = evaluate.load("f1")

In [43]:
precision_metric = evaluate.load("precision")

In [44]:
recall_metric = evaluate.load("recall")

In [45]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    metrics = {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)[
            "accuracy"
        ],
        "f1": f1_metric.compute(
            predictions=predictions, references=labels, average="macro"
        )["f1"],
        "precision": precision_metric.compute(
            predictions=predictions, references=labels, average="macro"
        )["precision"],
        "recall": recall_metric.compute(
            predictions=predictions, references=labels, average="macro"
        )["recall"],
    }

    return metrics

## Train the Model

In [46]:
# class CustomTrainer(Trainer):
#     def __init__(self, *args, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.train_losses = []
#         self.eval_losses = []

#     def training_step(self, model, inputs, return_loss=True):
#         loss = super().training_step(model, inputs, return_loss)
#         self.train_losses.append(loss.item())
#         return loss

#     def evaluation_step(self, model, inputs, return_loss=True):
#         loss = super().evaluation_step(model, inputs, return_loss)
#         self.eval_losses.append(loss.item())
#         return loss

In [47]:
trainer = Trainer( # CustomTrainer
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

In [51]:
%%wandb
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: stevillis (stevillis-sousa). Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.220000,0.212336,0.937117,0.936487,0.937309,0.936392
2,0.109700,0.212100,0.949387,0.949331,0.948499,0.951386
3,0.019900,0.199696,0.955521,0.955754,0.955914,0.955797


## Save and Upload the Model

In [53]:
torch.save(model.state_dict(), f"{PATH_TO_SAVE_MODEL}pytorch_model.bin")

In [54]:
model.save_pretrained(f"{PATH_TO_SAVE_MODEL}sentiment_model")

In [55]:
tokenizer.save_pretrained(f"{PATH_TO_SAVE_MODEL}sentiment_model")

('/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/train_model/sentiment_model/tokenizer_config.json',
 '/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/train_model/sentiment_model/special_tokens_map.json',
 '/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/train_model/sentiment_model/vocab.txt',
 '/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/train_model/sentiment_model/added_tokens.json',
 '/content/drive/MyDrive/UFMT/Gestão e Ciência de Dados/Disciplinas/14 - Seminário e Metodologia da Pesquisa/Projetos/glassdoor-reviews-analysis-nlp/train_model/sentiment_model/tokenizer.json')

In [56]:
notebook_login()  # Follow prompts to authenticate

In [58]:
model.push_to_hub("stevillis/bertimbau-finetuned-glassdoor-reviews")

README.md:   0%|          | 0.00/969 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/stevillis/bertimbau-finetuned-glassdoor-reviews/commit/e9519ae696f2418d14b04a9358aa7a4ce0cfb1f0', commit_message='Upload BertForSequenceClassification', commit_description='', oid='e9519ae696f2418d14b04a9358aa7a4ce0cfb1f0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/stevillis/bertimbau-finetuned-glassdoor-reviews', endpoint='https://huggingface.co', repo_type='model', repo_id='stevillis/bertimbau-finetuned-glassdoor-reviews'), pr_revision=None, pr_num=None)

In [59]:
tokenizer.push_to_hub("stevillis/bertimbau-finetuned-glassdoor-reviews")

CommitInfo(commit_url='https://huggingface.co/stevillis/bertimbau-finetuned-glassdoor-reviews/commit/8934d214814ef04dcab1ba742edec184170c04ff', commit_message='Upload tokenizer', commit_description='', oid='8934d214814ef04dcab1ba742edec184170c04ff', pr_url=None, repo_url=RepoUrl('https://huggingface.co/stevillis/bertimbau-finetuned-glassdoor-reviews', endpoint='https://huggingface.co', repo_type='model', repo_id='stevillis/bertimbau-finetuned-glassdoor-reviews'), pr_revision=None, pr_num=None)

## Inference Example

In [60]:
classifier = pipeline(
    "text-classification",
    model="stevillis/bertimbau-finetuned-glassdoor-reviews",
    return_all_scores=True,
)

config.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/678k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [61]:
for sample_review in SAMPLE_REVIEWS:
    result = classifier(sample_review)
    scores = result[0]
    highest_sentiment = max(scores, key=lambda x: x['score'])
    # must print positive, negative and neutral for the reviews
    print(f"Sample review: {sample_review}\nSentiment: {highest_sentiment['label']}, Score: {highest_sentiment['score']}\n\n")

Sample review: Empresa boa para trabalhar
Sentiment: positive, Score: 0.9982761144638062


Sample review: Empresa ruim para trabalhar
Sentiment: negative, Score: 0.9958627223968506


Sample review: Não tenho o que declarar
Sentiment: neutral, Score: 0.9971656203269958


